# In ra dữ liệu được hiện trên bảng (demo)

In [19]:
import requests
from pathlib import Path

BASE_URL = "https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx"

def fetch_and_dump_html(date_time_str="01/01/2022 14:00",
                        hc="2-3-4-76-77",
                        vm="",
                        lv=""):
    # 1) Chuẩn bị request
    params = {"td": date_time_str, "vm": vm, "lv": lv, "hc": hc}
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Cache-Control": "no-cache",
    }

    # 2) Gửi request
    r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding  # đảm bảo text decode đúng

    # 3) In thông tin yêu cầu
    print("[REQUEST] GET", BASE_URL)
    print("[REQUEST] params =", params)
    print("[INFO] Final URL    :", r.url)
    print("[INFO] Status code  :", r.status_code)
    print("[INFO] Redirects    :", len(r.history))
    if r.history:
        for i, h in enumerate(r.history, 1):
            print(f"  └─ [{i}] {h.status_code} -> {h.url}")

    # 4) Lưu toàn bộ HTML ra file (để đối chiếu)
    safe_dt = date_time_str.replace("/", "-").replace(":", "-").replace(" ", "_")
    out_path = Path(f"raw_embed_{safe_dt}_{hc}.html")
    out_path.write_text(r.text, encoding=r.encoding or "utf-8")
    print(f"[SAVE] HTML đã được lưu: {out_path.resolve()} (chars={len(r.text)})")

    # 5) IN RA TOÀN BỘ HTML TRÊN CONSOLE (có thể rất dài)
    print("\n[HTML_BEGIN]")
    print(r.text)
    print("[HTML_END]\n")

if __name__ == "__main__":
    # Gọi đúng tham số bạn đang dùng để so khớp với giao diện web đang xem
    fetch_and_dump_html("01/01/2022 14:00", "2-3-4-76-77")


[REQUEST] GET https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx
[REQUEST] params = {'td': '01/01/2022 14:00', 'vm': '', 'lv': '', 'hc': '2-3-4-76-77'}
[INFO] Final URL    : https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx?td=01%2F01%2F2022+14%3A00&vm=&lv=&hc=2-3-4-76-77
[INFO] Status code  : 200
[INFO] Redirects    : 0
[SAVE] HTML đã được lưu: C:\Users\Admin\OneDrive\A01. Giáo trình UIT\C01. HKI-2025-2026\Khai thác dữ liệu và ứng dụng - CS313.Q12\raw_embed_01-01-2022_14-00_2-3-4-76-77.html (chars=27818)

[HTML_BEGIN]

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head id="Head1"><title>
	Thông tin vận hành hồ chứa
</title><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta name="viewport" content="width=device-width, initial-scale=1.0" /><meta name="description" /><meta name="author" />
    <!-- Web Fonts -->
  

# Tiến hành crawl lại dữ liệu hiện tại

In [25]:
# -*- coding: utf-8 -*-
"""
Crawl trang embed EVN thủy điện:
- GET đúng tham số td=01/01/2022 14:00&vm=&lv=&hc=2-3-4-76-77
- Lưu HTML thô để đối chiếu
- Parse tất cả <table> -> DataFrame
- Bổ sung năm cho ô ngày 'dd/MM HH:mm' -> 'dd/MM/2022 HH:mm'
- Cắt bỏ đuôi 'Đồng bộ lúc: ...' (từ cụm đó đến hết chuỗi) ở mọi ô (đặc biệt cột 'Tên hồ')
- Xoá dòng header vùng (vd. 'Tây Bắc Bộ' bị lặp trên nhiều cột) & dòng 'Chú thích ký hiệu'
- Xuất CSV UTF-8-SIG
"""

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from typing import List

BASE_URL = "https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx"

# ==== Regex ====
# dd/MM HH:mm  -> thêm năm
RE_DDMM_HHMM = re.compile(r"\b(?P<dm>\d{2}/\d{2})\s+(?P<hm>\d{2}:\d{2})\b")
# Đồng bộ lúc: ... -> CẮT từ đây đến hết chuỗi (không phân biệt hoa/thường)
RE_CUT_SYNC_TAIL = re.compile(r"(?i)\s*đồng\s*bộ\s*lúc:.*$")

# ==== HTTP ====
def fetch_html(date_time_str="01/01/2022 14:00", hc="2-3-4-76-77", vm="", lv="") -> str:
    """Gửi request và trả về HTML (đồng thời log + lưu file)."""
    params = {"td": date_time_str, "vm": vm, "lv": lv, "hc": hc}
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Cache-Control": "no-cache",
    }
    r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding

    print("[REQUEST] GET", BASE_URL)
    print("[REQUEST] params =", params)
    print("[INFO] Final URL   :", r.url)
    print("[INFO] Status code :", r.status_code)
    print("[INFO] Redirects   :", len(r.history))

    # Lưu HTML
    safe_dt = date_time_str.replace("/", "-").replace(":", "-").replace(" ", "_")
    out_path = Path(f"raw_embed_{safe_dt}_{hc}.html")
    out_path.write_text(r.text, encoding=r.encoding or "utf-8")
    print(f"[SAVE] HTML -> {out_path.resolve()} (chars={len(r.text)})")

    return r.text

# ==== Parse ====
def parse_all_tables(html: str) -> List[pd.DataFrame]:
    """Ưu tiên pandas.read_html; nếu không có thì tự parse bằng bs4."""
    try:
        dfs = pd.read_html(html)  # list DataFrame
        if dfs:
            return dfs
    except ValueError:
        pass

    soup = BeautifulSoup(html, "html.parser")
    dfs: List[pd.DataFrame] = []
    for tb in soup.find_all("table"):
        rows = []
        for tr in tb.find_all("tr"):
            cells = [c.get_text(strip=True) for c in tr.find_all(["th", "td"])]
            if cells:
                rows.append(cells)
        if rows:
            try:
                # đoán header: mọi hàng cùng số cột và >1 hàng
                if len(rows) > 1 and len(set(map(len, rows))) == 1:
                    header, data = rows[0], rows[1:]
                    dfs.append(pd.DataFrame(data, columns=header))
                else:
                    dfs.append(pd.DataFrame(rows))
            except Exception:
                continue
    return dfs

# ==== Chuẩn hoá & Làm sạch ====
def normalize_ws(x):
    if not isinstance(x, str):
        return x
    return " ".join(x.split()).strip()

def add_year_to_cell(value: str, year: int) -> str:
    """Thêm năm 4 số vào chuỗi dạng 'dd/MM HH:mm' -> 'dd/MM/yyyy HH:mm'."""
    if not isinstance(value, str) or not value:
        return value
    value = normalize_ws(value)

    def _dmhm(m):
        return f"{m.group('dm')}/{year} {m.group('hm')}"
    value2 = RE_DDMM_HHMM.sub(_dmhm, value)
    return value2

def cut_sync_tail(value: str) -> str:
    """Xoá từ 'Đồng bộ lúc:' đến hết chuỗi."""
    if not isinstance(value, str) or not value:
        return value
    value = normalize_ws(value)
    value = RE_CUT_SYNC_TAIL.sub("", value)
    # gọt bỏ ký tự ngăn cách dư thừa
    return value.strip(" -–—|").strip()

def normalize_dates_in_df(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """Thêm năm cho mọi ô khớp 'dd/MM HH:mm'."""
    return df.applymap(lambda x: add_year_to_cell(x, year) if isinstance(x, str) else x)

def cut_sync_tail_df(df: pd.DataFrame) -> pd.DataFrame:
    """Cắt 'Đồng bộ lúc: ...' cho toàn bộ DataFrame (đặc biệt cột 'Tên hồ')."""
    return df.applymap(lambda x: cut_sync_tail(x) if isinstance(x, str) else x)

def drop_region_header_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Xoá các dòng header vùng (vd. toàn 'Tây Bắc Bộ' bị lặp trên nhiều cột).
    Quy tắc: trong các ô text KHÔNG RỖNG, nếu có >=2 ô và tất cả giống hệt nhau -> loại.
    """
    def is_region_row(row: pd.Series) -> bool:
        vals = []
        for v in row.values:
            if isinstance(v, str):
                s = normalize_ws(v)
                if s:
                    vals.append(s)
        if len(vals) >= 2 and len(set(vals)) == 1:
            return True
        return False

    mask = df.apply(is_region_row, axis=1)
    return df[~mask].reset_index(drop=True)

def drop_note_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Xoá dòng 'Chú thích ký hiệu' nếu tồn tại."""
    mask = df.astype(str).apply(
        lambda col: col.str.fullmatch(r"(?i)\s*chú thích ký hiệu\s*"), axis=0
    ).any(axis=1)
    return df[~mask].reset_index(drop=True)

# ==== Pipeline chính ====
def crawl_evnhydropower_to_csv(
    date_time_str="01/01/2022 14:00",
    hc="2-3-4-76-77",
    out_csv="ho_chua_thuy_dien_fix_2022.csv",
):
    # 1) Fetch
    html = fetch_html(date_time_str=date_time_str, hc=hc)

    # 2) Parse tất cả bảng → gộp
    dfs = parse_all_tables(html)
    if not dfs:
        raise RuntimeError("Không tìm thấy bảng HTML nào trong trang.")
    df_all = pd.concat(dfs, ignore_index=True, sort=False)

    # 3) Lấy năm mục tiêu từ tham số td
    m = re.search(r"(\d{4})", date_time_str)
    if not m:
        raise ValueError("Không tách được năm từ tham số td.")
    target_year = int(m.group(1))

    # 4) Chuẩn hoá ngày + cắt 'Đồng bộ lúc: ...'
    df_all = normalize_dates_in_df(df_all, target_year)
    df_all = cut_sync_tail_df(df_all)

    # 5) Xoá dòng header vùng + dòng chú thích (nếu có)
    df_all = drop_region_header_rows(df_all)
    df_all = drop_note_rows(df_all)

    # 6) Ghi CSV
    df_all.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[OK] Đã lưu CSV: {out_csv} (rows={len(df_all)})")
    return df_all

# ==== Run ====
if __name__ == "__main__":
    # Đúng link bạn yêu cầu: td=01/01/2022%2014:00&vm=&lv=&hc=2-3-4-76-77
    df = crawl_evnhydropower_to_csv(
        date_time_str="02/01/2022 14:00",
        hc="2-3-4-76-77",
        out_csv="ho_chua_thuy_dien_fix_2022.csv",
    )
    # Xem nhanh vài dòng đầu
    print(df.head(10))


[REQUEST] GET https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx
[REQUEST] params = {'td': '02/01/2022 14:00', 'vm': '', 'lv': '', 'hc': '2-3-4-76-77'}
[INFO] Final URL   : https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx?td=02%2F01%2F2022+14%3A00&vm=&lv=&hc=2-3-4-76-77
[INFO] Status code : 200
[INFO] Redirects   : 0
[SAVE] HTML -> C:\Users\Admin\OneDrive\A01. Giáo trình UIT\C01. HKI-2025-2026\Khai thác dữ liệu và ứng dụng - CS313.Q12\raw_embed_02-01-2022_14-00_2-3-4-76-77.html (chars=27851)
[OK] Đã lưu CSV: ho_chua_thuy_dien_fix_2022.csv (rows=5)
             Tên hồ         Thời điểm                      Htl  \
  Chú thích ký hiệu Chú thích ký hiệu Mực nước  thượng lưu (m)   
0          Bản Chát  02/01/2022 13:00                   474.23   
1        Huội Quảng  02/01/2022 13:00                   369.28   
2            Sơn La  02/01/2022 14:00                   214.96   
3          Hòa Bình  02/01/2022 13:00                   111.79   
4           Thác B

C:\Users\Admin\AppData\Local\Temp\ipykernel_8928\327438034.py:59: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)  # list DataFrame
C:\Users\Admin\AppData\Local\Temp\ipykernel_8928\327438034.py:113: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: add_year_to_cell(x, year) if isinstance(x, str) else x)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8928\327438034.py:117: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: cut_sync_tail(x) if isinstance(x, str) else x)


# Crawl dữ liệu từ 1/1/2022 đến 31/12/2024 theo các khung giờ

2:00 5:00 8:00 11:00 14:00 17:00 20:00 23:00

In [ ]:
# -*- coding: utf-8 -*-
"""
Crawl EVN hồ chứa thủy điện (2022-2024) theo khung giờ cố định.
- Tham số GET: td=dd/MM/yyyy HH:00&vm=&lv=&hc=2-3-4-76-77
- Làm sạch:
  + Thêm năm cho ô 'dd/MM HH:mm' -> 'dd/MM/yyyy HH:mm'
  + Xoá đuôi 'Đồng bộ lúc: ...' (từ cụm đó đến hết)
  + Xoá dòng header vùng (vd. 'Tây Bắc Bộ' lặp trên nhiều cột)
  + Xoá dòng 'Chú thích ký hiệu'
- Lưu nhanh bằng polars: ghi nhiều parquet part trong 1 thư mục (partitioned)
- Theo dõi tiến độ với tqdm
"""

import re
import time
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Iterable, Optional

import polars as pl
from tqdm import tqdm

BASE_URL = "https://hochuathuydien.evn.com.vn/PageHoChuaThuyDienEmbedEVN.aspx"
HC_PARAM = "2-3-4-76-77"
HOURS_PER_DAY = [2, 5, 8, 11, 14, 17, 20, 23]

# Thư mục output (partitioned parquet)
OUT_DIR = Path("evn_thuydien_2022_2024_parquet")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== Regex =====================
# dd/MM HH:mm  -> thêm năm vào giữa (dd/MM/yyyy HH:mm)
RE_DDMM_HHMM = r"(\d{2}/\d{2})\s+(\d{2}:\d{2})"
# Cắt từ 'Đồng bộ lúc:' đến hết chuỗi (không phân biệt hoa/thường)
RE_CUT_SYNC_TAIL = r"(?i)\s*đồng\s*bộ\s*lúc:.*$"

# ===================== HTTP =====================
def requests_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    })
    return s

def fetch_html(session: requests.Session, date_time_str: str, hc: str = HC_PARAM) -> str:
    params = {"td": date_time_str, "vm": "", "lv": "", "hc": hc}
    for attempt in range(4):  # retry nhẹ
        try:
            r = session.get(BASE_URL, params=params, timeout=30)
            r.raise_for_status()
            r.encoding = r.apparent_encoding
            return r.text
        except Exception:
            time.sleep(1.2 * (attempt + 1))
    raise RuntimeError(f"GET thất bại sau retry: td={date_time_str}")

# ===================== Parse (BeautifulSoup -> polars) =====================
def parse_all_tables_to_polars(html: str) -> List[pl.DataFrame]:
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    dfs: List[pl.DataFrame] = []

    for tb in tables:
        rows = []
        for tr in tb.find_all("tr"):
            cells = [c.get_text(strip=True) for c in tr.find_all(["th", "td"])]
            if cells:
                rows.append(cells)
        if not rows:
            continue

        # Đoán header: nếu mọi hàng cùng số cột và >1 hàng -> hàng đầu là header
        try:
            if len(rows) > 1 and len({len(r) for r in rows}) == 1:
                header, data = rows[0], rows[1:]
                df = pl.DataFrame(data, schema=[str(h) if h else f"c{i}" for i, h in enumerate(header)])
            else:
                df = pl.DataFrame(rows)
            dfs.append(df)
        except Exception:
            continue

    return dfs

# ===================== Cleaning (thuần polars, vectorized) =====================
def clean_polars_df(df: pl.DataFrame, year: int) -> pl.DataFrame:
    """
    - Thêm năm cho mẫu 'dd/MM HH:mm'
    - Cắt 'Đồng bộ lúc: ...' đến hết
    - Chuẩn hoá khoảng trắng/thừa
    - Xoá dòng header vùng và 'Chú thích ký hiệu'
    """
    # 1) Chuẩn hoá chuỗi & biến đổi trên các cột dạng Utf8
    utf8_cols = [c for c, dt in zip(df.columns, df.dtypes) if dt == pl.Utf8]

    # Thêm năm (regex backref) + cắt tail + gọn khoảng trắng/ký tự thừa
    # Dùng f-string để nhúng {year} vào replacement
    for c in utf8_cols:
        df = df.with_columns(
            pl.col(c)
              .str.replace_all(RE_DDMM_HHMM, rf"\1/{year} \2")   # dd/MM HH:mm -> dd/MM/yyyy HH:mm
              .str.replace_all(RE_CUT_SYNC_TAIL, "")             # cắt 'Đồng bộ lúc: ...'
              .str.replace_all(r"\s+", " ")                      # gọn whitespace
              .str.strip()
              .str.strip_chars_start("-–—| ")
              .str.strip_chars_end("-–—| ")
              .alias(c)
        )

    # 2) Xoá dòng 'Chú thích ký hiệu' (so sánh lowercase)
    # Tạo bản sao lowercase cho các cột Utf8, rồi any_horizontal
    lowered = [pl.col(c).str.to_lowercase().alias(f"__{c}_lc") for c in utf8_cols]
    df = df.with_columns(lowered)
    note_mask = pl.any_horizontal(
        *[pl.col(f"__{c}_lc") == "chú thích ký hiệu" for c in utf8_cols]
    )
    df = df.filter(~note_mask)
    df = df.drop([f"__{c}_lc" for c in utf8_cols])

    # 3) Xoá dòng header vùng: các ô text KHÔNG RỖNG có từ 2 ô trở lên và TẤT CẢ giống nhau
    # Việc này làm bằng Python (số dòng dạng này rất ít nên vẫn nhanh)
    def is_region_row(vals: list) -> bool:
        toks = [str(v).strip() for v in vals if isinstance(v, str) and str(v).strip()]
        return len(toks) >= 2 and len(set(toks)) == 1

    # Tạo mask bằng iter_rows (ghi nhớ: rất ít dòng bị xoá)
    mask_keep = []
    for row in df.iter_rows():
        mask_keep.append(not is_region_row(row))
    df = df.filter(pl.Series("keep", mask_keep))

    return df

# ===================== Core crawl =====================
def crawl_one_timestamp(session: requests.Session, dt: datetime, hc: str = HC_PARAM) -> Optional[pl.DataFrame]:
    td_str = dt.strftime("%d/%m/%Y %H:00")
    html = fetch_html(session, td_str, hc=hc)
    dfs = parse_all_tables_to_polars(html)
    if not dfs:
        return None
    df_all = pl.concat(dfs, how="vertical_relaxed")
    df_all = clean_polars_df(df_all, dt.year)
    # Gắn nhãn mốc thời gian yêu cầu
    df_all = df_all.with_columns(pl.lit(td_str).alias("td_query")).select(["td_query"] + df_all.columns)
    return df_all

# ===================== Lập lịch thời điểm =====================
def generate_datetimes(start_date: datetime, end_date: datetime, hours: Iterable[int]) -> List[datetime]:
    cur = datetime(start_date.year, start_date.month, start_date.day)
    end = datetime(end_date.year, end_date.month, end_date.day)
    out = []
    while cur <= end:
        for h in hours:
            out.append(cur.replace(hour=h, minute=0, second=0, microsecond=0))
        cur += timedelta(days=1)
    return out

# ===================== Ghi Parquet (partitioned) =====================
def write_parquet_part(df: pl.DataFrame, part_idx: int, out_dir: Path):
    # nén zstd để nhanh & gọn
    out_path = out_dir / f"part-{part_idx:06d}.parquet"
    df.write_parquet(out_path.as_posix(), compression="zstd")  # polars write: nhanh
    return out_path

# ===================== Pipeline tổng =====================
def crawl_range_to_parquet_parts(
    start_date: str = "01/01/2022",
    end_date: str = "31/12/2024",
    hours: Iterable[int] = HOURS_PER_DAY,
    hc: str = HC_PARAM,
    out_dir: Path = OUT_DIR,
    batch_size: int = 120,     # số mốc/đợt ghi
    polite_delay: float = 0.6  # delay giữa các request
):
    # Dọn thư mục output (tuỳ ý). Nếu muốn append, bỏ đoạn xoá.
    for p in out_dir.glob("part-*.parquet"):
        p.unlink(missing_ok=True)

    sess = requests_session()

    sd = datetime.strptime(start_date, "%d/%m/%Y")
    ed = datetime.strptime(end_date, "%d/%m/%Y")
    dts = generate_datetimes(sd, ed, hours)
    total = len(dts)

    batch_frames: List[pl.DataFrame] = []
    part_idx = 0

    with tqdm(total=total, desc="Crawling EVN (polars)", unit="ts") as pbar:
        for dt in dts:
            try:
                df = crawl_one_timestamp(sess, dt, hc=hc)
                if df is not None and df.height > 0:
                    batch_frames.append(df)
            except Exception:
                # Bỏ qua mốc lỗi để tiếp tục
                pass

            if len(batch_frames) >= batch_size:
                combined = pl.concat(batch_frames, how="vertical_relaxed")
                write_parquet_part(combined, part_idx, out_dir)
                part_idx += 1
                batch_frames = []

            pbar.update(1)
            time.sleep(polite_delay)

    # ghi nốt phần còn lại
    if batch_frames:
        combined = pl.concat(batch_frames, how="vertical_relaxed")
        write_parquet_part(combined, part_idx, out_dir)

    print(f"[DONE] Đã ghi các part vào: {out_dir.resolve()}")
    print("Cách đọc/gộp nhanh bằng polars:")
    print(f'  df = pl.scan_parquet("{out_dir.as_posix()}/*.parquet").collect()')

# ===================== Run =====================
if __name__ == "__main__":
    crawl_range_to_parquet_parts(
        start_date="01/01/2022",
        end_date="31/12/2024",
        hours=[2,5,8,11,14,17,20,23],
        hc="2-3-4-76-77",
        out_dir=OUT_DIR,
        batch_size=120,
        polite_delay=0
    )
